In [1]:
import os

In [3]:
!pip install -q langchain-openai langchain-core requests

In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [5]:
from langchain_core.tools import InjectedToolArg
from typing import Annotated

In [6]:
# tool create
@tool
def multiply(a:int,b:int) -> int :
    """Given 2 numbers a and b this tool returns their product"""
    return a*b

In [7]:
multiply.invoke({"a":3,"b":4})

12

In [8]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Given 2 numbers a and b this tool returns their product
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [9]:
# tool binding

In [10]:
llm = ChatOpenAI(model="gpt-6-luna")

In [11]:
llm_with_tools = llm.bind_tools([multiply])

In [12]:
llm_with_tools.invoke("Hi, how are you?")

AIMessage(content=[{'type': 'text', 'text': 'Hi! I’m doing well, thanks for asking. How are you?', 'annotations': [], 'id': 'msg_05073cba68ec20ed006b0356ee33a487d2a3c17583c111a78f', 'phase': 'final_answer'}], additional_kwargs={}, response_metadata={'id': 'resp_05073cba68ec20ed006b0356ed804487d28b50045042efd0b2', 'created_at': 1795380973.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_05073cba68ec20ed006b0356ed804487d28b50045042efd0b2', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 19, 'total_tokens': 77, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}})

In [13]:
query = HumanMessage("Can you miltiply 3 with 10")

In [14]:
messages = [query]

In [15]:
messages

[HumanMessage(content='Can you miltiply 3 with 10', additional_kwargs={}, response_metadata={})]

In [16]:
result = llm_with_tools.invoke(messages)

In [17]:
result

AIMessage(content=[{'arguments': '{"a":3,"b":10}', 'call_id': 'call_98e2d563bff94fdd9590f6fe30dbaeb6', 'name': 'multiply', 'type': 'function_call', 'id': 'fc_04cc03101cdee9f5006b0356ef608087d29f83d07bb5eed9a8', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_04cc03101cdee9f5006b0356eef4d887d282efff6c70642d32', 'created_at': 1795380975.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_04cc03101cdee9f5006b0356eef4d887d282efff6c70642d32', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'call_98e2d563bff94fdd9590f6fe30dbaeb6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 62, 'output_tokens': 21, 'total_tokens': 83, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}})

In [18]:
messages.append(result)

In [19]:
result.tool_calls

[{'name': 'multiply',
  'args': {'a': 3, 'b': 10},
  'id': 'call_98e2d563bff94fdd9590f6fe30dbaeb6',
  'type': 'tool_call'}]

In [20]:
tool_result = multiply.invoke(result.tool_calls[0])

In [21]:
messages.append(tool_result)

In [22]:
messages

[HumanMessage(content='Can you miltiply 3 with 10', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[{'arguments': '{"a":3,"b":10}', 'call_id': 'call_98e2d563bff94fdd9590f6fe30dbaeb6', 'name': 'multiply', 'type': 'function_call', 'id': 'fc_04cc03101cdee9f5006b0356ef608087d29f83d07bb5eed9a8', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_04cc03101cdee9f5006b0356eef4d887d282efff6c70642d32', 'created_at': 1795380975.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_04cc03101cdee9f5006b0356eef4d887d282efff6c70642d32', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 10}, 'id': 'call_98e2d563bff94fdd9590f6fe30dbaeb6', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 62, 'output_tokens': 21, 'total_tokens': 83, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_t

In [23]:
llm_with_tools.invoke(messages).content

[{'type': 'text',
  'text': '3 × 10 = 30.',
  'annotations': [],
  'id': 'msg_04cc03101cdee9f5006b0356f050f087d28a6c61fff8d20220',
  'phase': 'final_answer'}]

In [24]:
# tool create
@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:

    """Get the exchange rate from the base currency to the target currency."""

    url = f"https://v6.exchangerate-api.com/v6/{os.environ['EXCHANGE_RATE_API_KEY']}/pair/{base_currency}/{target_currency}"

    response = requests.get(url)

    return response.json()


@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float,InjectedToolArg]) -> float:

    """Convert an amount from one currency to another using the exchange rate."""

    return base_currency_value * conversion_rate

In [25]:
get_conversion_factor.invoke({"base_currency":"USD","target_currency":"INR"})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1795305601,
 'time_last_update_utc': 'Sun, 22 Nov 2026 00:00:01 +0000',
 'time_next_update_unix': 1795392001,
 'time_next_update_utc': 'Mon, 23 Nov 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 92.8891}

In [26]:
convert.invoke({"base_currency_value":10,"conversion_rate":92.8891})

928.891

In [27]:
llm = ChatOpenAI(model="gpt-6-luna")

In [28]:
llm_with_tools = llm.bind_tools([get_conversion_factor,convert])

In [29]:
messages = [HumanMessage(
    content="What is the conversion factor between USD and INR and based on that can you convert 10 USD to INR?"
)]

In [30]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR and based on that can you convert 10 USD to INR?', additional_kwargs={}, response_metadata={})]

In [31]:
ai_message = llm_with_tools.invoke(messages)

In [32]:
messages.append(ai_message)

In [33]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'INR'},
  'id': 'call_0412f1394a60428c821fa13782d3de0a',
  'type': 'tool_call'}]

In [34]:
tool_message1= get_conversion_factor.invoke(ai_message.tool_calls[0])

In [35]:
tool_message1

ToolMessage(content='{"result": "success", "documentation": "https://www.exchangerate-api.com/docs", "terms_of_use": "https://www.exchangerate-api.com/terms", "time_last_update_unix": 1795305601, "time_last_update_utc": "Sun, 22 Nov 2026 00:00:01 +0000", "time_next_update_unix": 1795392001, "time_next_update_utc": "Mon, 23 Nov 2026 00:00:01 +0000", "base_code": "USD", "target_code": "INR", "conversion_rate": 92.8891}', name='get_conversion_factor', tool_call_id='call_0412f1394a60428c821fa13782d3de0a')

In [36]:
import json

data = json.loads(tool_message1.content)

conversion_rate = data["conversion_rate"]

print(conversion_rate)

92.8891


In [37]:
messages.append(tool_message1)

In [38]:
ai_message2 = llm_with_tools.invoke(messages)

In [39]:
messages.append(ai_message2)

In [40]:
tool_call = ai_message2.tool_calls[0]

In [41]:
tool_call["args"]["conversion_rate"] = conversion_rate

In [42]:
tool_message2 = convert.invoke(tool_call)

In [43]:
tool_message2

ToolMessage(content='928.891', name='convert', tool_call_id='call_odfdnwGXfxwfSRPQlusDcpa2')

In [44]:
messages.append(tool_message2)

In [45]:
messages

[HumanMessage(content='What is the conversion factor between USD and INR and based on that can you convert 10 USD to INR?', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[{'id': 'rs_0b28d63177d97cff006b0356f187e087d2b60278f5cd3aac3e', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABrA1bxQ0F-ZDwFzwXa5QDd5m2V7eLFzCAz6SZO64_ZRMjQAIPL-hoCu1-67N66t4j4_L1uJmFsJ2v9xbxP-ZJzBqr-VJEuSRhyT0_1EstGPo-8Ih0dZB_gOrtKzBLG2P2-vM5H2_NSRB76r5VRrhPTy6ZgyZSnAGu1rbKDLikIPQIyu-yn9VZlELu3GJKbiiPH45lhcngUPxP1NQwAIzUsH7-hFhz0QQMWp6hGH_xUdbsTiNFXxSL37VUpMnXvjTjh95Byj8lHHllvy7qSUdYylaNNVZYHbt2lGE0WJolpBBAiV_BBhvA3HfCsyEYvjJTlUreuwZDYv2gC2YHIutxa3nUT5wvYSPzPhoF-Flqs9cc1D1H4oV1yFJ42nAfEGWsFxRjzOfw311Bc2QBUJi8OKDIGEYHoq0_EDczggOS5FghTBIURirbn1coS1eeXUK6w-G5xqXFgxePiLsrqK_sUD0RosBEKqD8I5ZmTPTRN5o2Qq56IvLuq8gSPLzHrxOxl5HUsVPC5EJe0VejVe3Pu59Nu6r2XtP_uu2SHvwU3s_c9oCbJ4P3RShdlB4FmJ13TddvWrg0JcQD4CxBLg453fooilt8lX5ylW7nNO5qUytxO9vAltxARlmfyoJxczkW8MTifvfkHLkEltSPGCIdfwLhLfp

In [48]:
answer = llm_with_tools.invoke(messages).content

In [49]:
final_answer = answer[-1]["text"]

print(final_answer)

The conversion factor is **1 USD = 92.8891 INR**.  
So **10 USD = 928.891 INR**, or about **₹928.89**.
